# [생활] 지하철 실시간 도착 · 출퇴근 안내 챗봇

**data-eng 7주차 RAG 챗봇 콘테스트 · 예시 노트북 05** — 오늘 배운 **Document Loader → 텍스트 분할 → Chunk 파라미터 튜닝 · 품질 비교**를 그대로 따라 만든 완성형 챗봇입니다.

### 시작하는 방법
1. 「Drive로 복사」 → 메뉴 **런타임 → 모두 실행**. ⑥ 셀 아래(또는 `https://….gradio.live`)에 챗봇이 뜹니다.
2. **키 없이도 끝까지 돌아갑니다**: 검색은 TF-IDF, 답은 찾은 조각을 그대로 보여 줍니다.
3. 코랩 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 를 넣으면 7주차와 같은 `text-embedding-3-small` 임베딩 검색 + `gpt-4o-mini` 답변이 됩니다.

### 이 예시가 쓰는 외부 API
- **서울 열린데이터광장 지하철 실시간 도착정보 API**
- 이 예시의 키 이름: `SEOUL_KEY` (선택) — 키 없이도 서울시 시험용 키 sample 로 5건을 받습니다. 더 많이 받으려면 data.seoul.go.kr → 인증키 신청(무료).

### 콘테스트 기준과 셀 대응
| 셀 | 오늘 배운 것 | 심사 항목 |
|---|---|---|
| ③ Loader | API 응답 → `Document`(metadata 에 출처 · 받은 시각) | Loader · 외부 API 20 |
| ④ 분할기 비교 | 문자 · 재귀 · 제목 우선 분할기를 같은 크기로 비교 | 분할기 선택 15 |
| ⑤ 파라미터 튜닝 | chunk_size · overlap 4가지 × 질문 3개 적중표 | **파라미터 튜닝 · 품질 비교 30** |
| ⑥ 완성 | 고른 설정으로 검색 → 근거 붙은 답, Gradio | 완성도 20 · 쓸모 · 신뢰성 15 |

## ① 설치 (1분쯤)

In [ ]:
!pip install -q -U gradio openai langchain-core langchain-text-splitters pandas scikit-learn
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② 키 불러오기 (없어도 됩니다)

In [ ]:
TOPIC_KEY = "SEOUL_KEY"

import os, time
os.environ["TZ"] = "Asia/Seoul"      # 코랩 시계는 UTC — 날짜 · 받은 시각을 한국 기준으로
time.tzset()
NAMES = ["OPENAI_API_KEY"] + ([TOPIC_KEY] if TOPIC_KEY else [])
try:
    from google.colab import userdata          # 코랩 왼쪽 🔑 보안 비밀
    for n in NAMES:
        try:
            v = userdata.get(n)
            if v:
                os.environ[n] = v
        except Exception:
            pass
except ImportError:
    pass
for n in NAMES:
    print(f"{n:16s}", "✅ 있음" if os.environ.get(n) else "— 없음")
print("\nOpenAI 키가 있으면 임베딩 검색 + AI 답변, 없으면 TF-IDF 검색 + 찾은 조각을 그대로 보여 줍니다.")

## ③ Document Loader — 외부 API 데이터를 Document 로
안내 문서 1개 + API 에서 받은 데이터 1개, 모두 `Document` 로 만듭니다. `metadata` 에 **출처와 받은 시각**을 남겨 두면 답에 근거로 붙일 수 있습니다.
API 를 부르지 못하면(키 없음 · 네트워크) **예시 데이터**로 대신하고, 출처에 그렇게 표시합니다.

In [ ]:
import os, re, json, datetime as dt
import requests
UA = {"User-Agent": "colab-rag-contest-example/1.0"}

TITLE = "🚇 지하철 도착 도우미"
ROLE = "서울 지하철 실시간 도착 정보와 출퇴근 요령을 알려 주는 생활 도우미"
TOPIC_KEY = "SEOUL_KEY"
OPTIONS = {"역 이름": "서울"}
HOWTO = "역 이름을 바꾸고 「데이터 새로 받기」를 누르면 그 역의 도착 정보를 받아 옵니다. 실시간 정보라 자주 새로 받으세요."
QUESTIONS = [("다음 열차 언제 와?", ["도착"]),
             ("2호선 노선 ID는?", ["1002"]),
             ("실시간 정보는 얼마나 자주 바뀌어?", ["새로 받"])]

GUIDE_SOURCE = "지하철 이용 안내(이 노트북의 안내 문서)"
GUIDE = """# 지하철 이용 안내
## 1. 노선 ID (서울시 API 의 subwayId)
1001은 1호선, 1002는 2호선, 1003은 3호선, 1004는 4호선, 1005는 5호선, 1006은 6호선, 1007은 7호선, 1008은 8호선, 1009는 9호선입니다. 1063은 경의중앙선, 1065는 공항철도, 1075는 수인분당선, 1077은 신분당선입니다.
## 2. 실시간 정보 읽기
도착 정보는 열차 위치에 따라 계속 바뀝니다. 출발 직전에 「데이터 새로 받기」로 다시 받아 확인합니다. 「전역 도착」은 한 정거장 전에 도착했다는 뜻이고, 「진입」은 지금 들어오고 있다는 뜻입니다.
## 3. 출퇴근 요령 (이 챗봇의 안내)
출근 시간에는 2~3분 간격이라도 붐비므로 한 대를 보내고 다음 열차를 타는 것도 방법입니다. 환승역에서는 환승 통로 시간을 5분 정도로 잡습니다."""

SAMPLE = """# 서울역 실시간 도착 (예시 데이터)
## 소요산행 - 시청방면
1호선 상행 열차가 전역 도착 상태입니다(남영). 정보 생성 시각 2026-11-09 18:00:00."""

LINES = {"1001": "1호선", "1002": "2호선", "1003": "3호선", "1004": "4호선", "1005": "5호선", "1006": "6호선",
         "1007": "7호선", "1008": "8호선", "1009": "9호선", "1063": "경의중앙선", "1065": "공항철도",
         "1075": "수인분당선", "1077": "신분당선"}

def fetch_api(**opt):
    st = (opt.get("역 이름") or "서울").strip().removesuffix("역")
    key = os.environ.get("SEOUL_KEY") or "sample"
    n = 5 if key == "sample" else 20
    j = requests.get(f"http://swopenapi.seoul.go.kr/api/subway/{key}/json/realtimeStationArrival/0/{n}/{st}", timeout=20).json()
    rows = j.get("realtimeArrivalList") or []
    if not rows:
        raise RuntimeError(j.get("message") or j.get("errorMessage", {}).get("message") or "도착 정보가 없습니다")
    lines = [f"# {st}역 실시간 도착"]
    for r in rows:
        line = LINES.get(str(r.get("subwayId")), f"노선 {r.get('subwayId')}")
        lines.append(f"## {r.get('trainLineNm')}\n{line} {r.get('updnLine')} 열차가 {r.get('arvlMsg2')} 상태입니다"
                     f"({r.get('arvlMsg3')}). 정보 생성 시각 {r.get('recptnDt')}.")
    return "\n\n".join(lines), f"서울 열린데이터광장 실시간 도착({'시험용 키' if key == 'sample' else '인증키'})"

# ---------- Document Loader: 안내 문서 1개 + API 데이터 1개 ----------
from langchain_core.documents import Document

def now():
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M")

def load_docs(**opt):
    docs = [Document(page_content=GUIDE, metadata={"source": GUIDE_SOURCE, "fetched_at": "노트북에 포함"})]
    try:
        text, src = fetch_api(**opt)
        docs.append(Document(page_content=text, metadata={"source": src, "fetched_at": now()}))
    except Exception as e:
        print("⚠️ API 를 부르지 못해 예시 데이터로 대신합니다 →", e)
        docs.append(Document(page_content=SAMPLE, metadata={"source": "예시 데이터(응답 모양만 같음 · 실제 값 아님)",
                                                            "fetched_at": now()}))
    return docs

docs = load_docs(**OPTIONS)
for d in docs:
    print(f"📄 {d.metadata['source']} · {d.metadata['fetched_at']} · {len(d.page_content):,}자")
    print(d.page_content[:400], "\n…\n")

## ④ 분할기 비교 — 같은 크기(500 / 50)로 세 가지
- **문자**: 빈 줄 하나로만 자릅니다. 빈 줄 사이가 길면 조각이 크기를 넘습니다(최대 길이 확인).
- **재귀**: 제목 → 문단 → 줄 → 문장 → 낱말 순서로 시도해 크기를 지킵니다.
- **제목 우선**: `##` 제목 경계로 먼저 나눠 제목과 본문이 같은 조각에 들어갑니다.
「중간 절단」이 적고 조각 크기가 고른 분할기를 고르고, ⑤ 셀의 `SPLITTER` 에 적습니다.

In [ ]:
import re, statistics
import pandas as pd
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

# 마침표 · 「다」 등으로 끝나지 않는 조각 = 문장 중간에서 잘린 조각(오늘 수업 broken.py 와 같은 기준)
END = re.compile(r"([.!?。」)\]]|다|요|음|함|임)\s*$")

def stats(chunks):
    L = [len(c.page_content) for c in chunks]
    cut = sum(1 for c in chunks if not END.search(c.page_content.strip()))
    return {"조각 수": len(L), "평균 길이": round(statistics.mean(L)) if L else 0,
            "최대 길이": max(L) if L else 0, "중간 절단": cut}

def split_char(docs, size, overlap):        # 구분자 하나(빈 줄)로만 자름
    return CharacterTextSplitter(separator="\n\n", chunk_size=size, chunk_overlap=overlap).split_documents(docs)

def split_rec(docs, size, overlap):         # 제목 → 문단 → 줄 → 문장 → 낱말 순서로 시도
    return RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap,
                                          separators=["\n## ", "\n\n", "\n", ". ", " ", ""]).split_documents(docs)

def split_md(docs, size, overlap):          # 먼저 ## 제목 경계로 나누고, 긴 것만 다시 재귀 분할
    md = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2")], strip_headers=False)
    parts = []
    for d in docs:
        for p in md.split_text(d.page_content):
            p.metadata = {**d.metadata, **p.metadata}
            parts.append(p)
    return RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap).split_documents(parts)

SPLITTERS = {"문자(Character)": split_char, "재귀(RecursiveCharacter)": split_rec, "제목 우선(MarkdownHeader+재귀)": split_md}

import warnings; warnings.filterwarnings("ignore")      # 문자 분할기의 「조각이 너무 큼」 경고는 표의 최대 길이로 봅니다
rows = [{"분할기": n, **stats(f(docs, 500, 50))} for n, f in SPLITTERS.items()]
display(pd.DataFrame(rows))
first = split_md(docs, 500, 50)[:2]
for c in first:
    print("──", c.metadata.get("h2", ""), "\n", c.page_content[:200], "\n")

## ⑤ Chunk 파라미터 튜닝 · 품질 비교
같은 질문 3개로 네 가지 설정을 비교합니다. 표의 칸은 이렇게 읽습니다.
- **질문1~3**: 질문의 핵심어가 든 조각이 검색 결과 몇 위로 나왔는지(✗ = 3위 안에 없음)
- **1위 적중**: 1위로 맞게 찾은 질문 수
- **AI에 보내는 글자**: 상위 3조각의 글자 수. 크게 자를수록 쉽게 맞히지만 **비용과 잡음이 늘어납니다**
- **중간 절단**: 문장 중간에서 잘린 조각 수

**고르는 규칙**: 1위 적중이 가장 많은 설정 중에서 「보내는 글자 + 중간 절단 1개당 100자 벌점」이 가장 작은 것. 마음에 들지 않으면 `BEST = (800, 80)` 처럼 직접 정해도 됩니다.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from openai import OpenAI

client = OpenAI() if os.environ.get("OPENAI_API_KEY") else None
EMB_MODEL, CHAT_MODEL = "text-embedding-3-small", "gpt-4o-mini"      # 7주차 수업과 같은 모델
_cache = {}

def embed(texts):
    need = [t for t in dict.fromkeys(texts) if t not in _cache]
    for i in range(0, len(need), 100):
        r = client.embeddings.create(model=EMB_MODEL, input=need[i:i + 100])
        for t, d in zip(need[i:i + 100], r.data):
            _cache[t] = np.array(d.embedding)
    M = np.array([_cache[t] for t in texts])
    return M / np.linalg.norm(M, axis=1, keepdims=True)

class Index:
    """조각 목록으로 검색기를 만듭니다. 키가 있으면 임베딩, 없으면 TF-IDF(글자 2~3개)."""
    def __init__(self, chunks):
        self.chunks = chunks
        texts = [c.page_content for c in chunks]
        if client:
            self.mode, self.M = "OpenAI 임베딩", embed(texts)
        else:
            self.mode = "TF-IDF(키 없음)"
            self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
            self.M = self.vec.fit_transform(texts)
    def search(self, q, k=3):
        s = self.M @ embed([q])[0] if client else (self.M @ self.vec.transform([q]).T).toarray().ravel()
        return [(self.chunks[i], float(s[i])) for i in np.argsort(-s)[:k]]

SPLITTER = "재귀(RecursiveCharacter)"           # ← 위 표를 보고 고른 분할기로 바꿔 보세요
SETTINGS = [(200, 0), (400, 40), (800, 80), (1500, 150)]   # (chunk_size, chunk_overlap) — overlap 은 보통 10~20%

rows = []
for size, ov in SETTINGS:
    chunks = SPLITTERS[SPLITTER](docs, size, ov)
    idx = Index(chunks)
    r = {"설정": f"{size} / {ov}", **stats(chunks)}
    first, sent = 0, 0
    for qi, (q, kws) in enumerate(QUESTIONS, 1):
        top = idx.search(q, 3)
        rank = next((i for i, (c, _) in enumerate(top, 1) if any(k in c.page_content for k in kws)), None)
        r[f"질문{qi}"] = f"{rank}위" if rank else "✗"          # 핵심어가 든 조각이 몇 번째로 찾아졌나
        first += rank == 1
        sent += sum(len(c.page_content) for c, _ in top)
    r["1위 적중"] = first
    r["AI에 보내는 글자(질문당)"] = sent // len(QUESTIONS)      # 크게 자를수록 많이 보냄 = 비용 · 잡음
    rows.append(r)

table = pd.DataFrame(rows)
display(table)
# 고르는 규칙: ① 1위 적중이 많은 설정 중 ② 「보내는 글자 + 중간 절단 1개당 100자 벌점」이 가장 작은 설정
best = max(rows, key=lambda r: (r["1위 적중"], -(r["AI에 보내는 글자(질문당)"] + 100 * r["중간 절단"])))
BEST = tuple(int(x) for x in best["설정"].split(" / "))
print(f"검색 방식: {idx.mode}")
for i, (q, kws) in enumerate(QUESTIONS, 1):
    print(f"질문{i}: {q}  (핵심어 {', '.join(kws)})")
print(f"\n→ 고른 설정 {BEST[0]} / {BEST[1]} — 1위 적중 {best['1위 적중']}/{len(QUESTIONS)} · "
      f"질문당 {best['AI에 보내는 글자(질문당)']:,}자 · 중간 절단 {best['중간 절단']} · 조각 {best['조각 수']}개")
print("✍️ 왜 이 설정인지 아래 칸에 한두 줄 적어 두세요(콘테스트 심사 30점).")

### ✍️ 왜 이 설정인지 (내 말로 적기 — 심사 30점)
- 고른 분할기와 이유:
- 고른 chunk_size / overlap 과 이유:
- 작게(200) 잘랐을 때 생긴 문제:
- 크게(1500) 잘랐을 때 생긴 문제:

## ⑥ 완성 — 챗봇 실행
멈추려면 ■(중지). 질문 칸 아래 예시를 눌러 보세요.

In [ ]:
import inspect
import gradio as gr

def to_text(c):
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return "\n".join(x.get("text", "") if isinstance(x, dict) else str(x) for x in c)
    return str(c or "")

SYSTEM = f"""너는 {ROLE}야. 아래 [자료]만 근거로 한국어로 짧고 쉽게 답해.
- 숫자 · 날짜 · 이름은 자료에 적힌 그대로 써. 자료에 [계산] 이 있으면 그 값을 써.
- 자료에 없는 내용은 「받아 온 자료에서 찾지 못했어요」라고 말하고 지어내지 마.
- 답 끝에 「근거:」 다음에 자료의 출처와 기준 시각을 적어."""

STATE = {"docs": docs, "index": Index(SPLITTERS[SPLITTER](docs, *BEST))}

def refresh(*vals):
    opt = dict(zip(OPTIONS.keys(), vals))
    STATE["docs"] = load_docs(**opt)
    STATE["index"] = Index(SPLITTERS[SPLITTER](STATE["docs"], *BEST))
    d = STATE["docs"][-1].metadata
    return f"✅ 새로 받았습니다 — {d['source']} · {d['fetched_at']} · 조각 {len(STATE['index'].chunks)}개"

def respond(q, history):
    q = (q or "").strip()
    if not q:
        raise gr.Error("질문을 적어 주세요.")
    history = history or []
    hits = STATE["index"].search(q, 4)
    ctx = "\n\n".join(f"<<{c.metadata.get('source')} · {c.metadata.get('fetched_at')}>>\n{c.page_content}" for c, _ in hits)
    extra = EXTRA(q) if callable(globals().get("EXTRA")) else ""
    if client:
        msgs = [{"role": "system", "content": SYSTEM}]
        msgs += [{"role": m["role"], "content": to_text(m["content"])[:2000]} for m in history[-4:]]
        msgs.append({"role": "user", "content": f"[자료]\n{ctx}\n{extra}\n\n[질문]\n{q}"})
        try:
            ans = client.chat.completions.create(model=CHAT_MODEL, messages=msgs).choices[0].message.content
        except Exception as e:
            s = str(e)
            raise gr.Error("OpenAI 키가 맞지 않습니다." if "api_key" in s or "401" in s else
                           "사용 한도를 넘었습니다. 잠시 뒤 다시 해 주세요." if "429" in s or "quota" in s else f"오류: {s[:200]}")
    else:
        ans = "(OpenAI 키가 없어 AI 답 대신 찾은 조각을 보여 드립니다)\n\n" + (extra + "\n\n" if extra else "") + \
              "\n\n".join(f"**{i}.** " + c.page_content[:300] for i, (c, _) in enumerate(hits, 1))
    src = "\n".join(sorted({f"- {c.metadata.get('source')} · {c.metadata.get('fetched_at')}" for c, _ in hits}))
    ans += f"\n\n---\n**찾은 조각의 출처** (설정 {BEST[0]}/{BEST[1]} · {STATE['index'].mode})\n{src}"
    return "", history + [{"role": "user", "content": q}, {"role": "assistant", "content": ans}]

chat_kw = {"height": 440}
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:      # Gradio 4~5
    chat_kw["type"] = "messages"

with gr.Blocks(title=TITLE) as demo:
    gr.Markdown(f"# {TITLE}\n{HOWTO}")
    with gr.Row():
        boxes = [gr.Textbox(label=k, value=str(v)) for k, v in OPTIONS.items()]
        btn = gr.Button("데이터 새로 받기", variant="secondary")
    status = gr.Markdown(f"데이터: {docs[-1].metadata['source']} · {docs[-1].metadata['fetched_at']}")
    chat = gr.Chatbot(label="대화", **chat_kw)
    q = gr.Textbox(label="질문", placeholder=QUESTIONS[0][0])
    gr.Examples([x[0] for x in QUESTIONS], q)
    btn.click(refresh, boxes, status)
    q.submit(respond, [q, chat], [q, chat])

demo.launch(share=True, debug=True)

## 제출 전 체크리스트 (패들릿 「콘테스트 제출」 칸)
- [ ] 위 「왜 이 설정인지」 칸을 내 말로 채웠다
- [ ] 코랩 공유 → 「링크가 있는 모든 사용자 · 뷰어」
- [ ] 패들릿 글: **이름 — 챗봇 이름** / ① 코랩 링크 ② 푸는 문제 한두 줄 ③ 쓴 API · 분할기 ④ 고른 chunk 설정과 이유 ⑤ 화면 · 비교표 캡처

## 내 작품으로 바꾸려면
1. **③ Loader 셀**의 `fetch_api` 를 내 API 로, `GUIDE` 를 내 안내 문서로 바꿉니다.
2. `QUESTIONS` 의 질문 3개와 핵심어를 내 주제에 맞게 바꿉니다(비교표가 이 질문으로 채점됩니다).
3. ④ · ⑤ 셀을 다시 실행해 분할기와 chunk 설정을 고르고, 이유를 적습니다.
4. ⑥ 셀의 `ROLE` · `TITLE` 을 바꾸고 실행합니다.